# All Alarm Tags — Events & Control Actions Analysis

End-to-end pipeline covering the 25 UC3 alarm `(tag, condition)` pairs:

1. **Setup** — paths, discover combined-events parquet files  
2. **Alarm Tags & Data File Mapping** — map each `(tag, condition)` to its combined-events file  
3. **Loading & Validating Combined Events Data** — cache, inspect schema, verify alarm event counts  
4. **PV/OP Data — Missing Value Analysis** — data-quality check for each PV/OP parquet in `DATA/PV-OP_data/`  
5. **Alarm Episode Extraction** — pair Start → OK events into episodes; duration summary; filter significant  
6. **Control Actions Extraction** — pull operator CHANGE events (240 min before → 60 min after each alarm)  
7. **Control Actions Analysis** — classify action types; top operated tags per alarm  
8. **Data Coverage — Timestamp Ranges** — date span of each combined-events file  
9. **Export to Excel** — per-tag missing-value workbook and per-alarm control-actions workbook

## 1. Setup

Resolve project paths and discover available combined-events parquet files.

In [19]:
import pandas as pd
from pathlib import Path

# ── Paths ───────────────────────────────────────────────────────────────────────
PROJECT_ROOT        = Path('/home/h604827/ControlActions')
DATA_DIR            = PROJECT_ROOT / 'DATA'
COMBINED_EVENTS_DIR = DATA_DIR / 'combined_events'

print(f"Combined events dir : {COMBINED_EVENTS_DIR}")
available_files = sorted(COMBINED_EVENTS_DIR.glob('*_combined_events.parquet'))
print(f"Available files     : {len(available_files)}")
for f in available_files:
    mb = f.stat().st_size / (1024 ** 2)
    print(f"  {f.name:<60s}  {mb:6.1f} MB")


Combined events dir : /home/h604827/ControlActions/DATA/combined_events
Available files     : 17
  03FIC_1668_PVHI_combined_events.parquet                        103.3 MB
  03LIC1608_PVLO_combined_events.parquet                          83.4 MB
  03LIC_1016_PVLO_PVHI_combined_events.parquet                   113.4 MB
  03LIC_1071_PVLO_PVHI_combined_events.parquet                   146.4 MB
  03LIC_1619_PVLO_PVHI_combined_events.parquet                    76.1 MB
  03PIC1023_PVLO_PVHI_combined_events.parquet                    119.5 MB
  03PIC_1013_PVLO_combined_events.parquet                         43.0 MB
  03PIC_1104_PVHI_combined_events.parquet                        154.6 MB
  03PI_1655_PVHI_combined_events.parquet                          20.6 MB
  03TIC_1009_PVHI_combined_events.parquet                         98.3 MB
  03TIC_1009_PVLO_combined_events.parquet                         98.3 MB
  03TIC_1023_PVLO_PVHI_combined_events.parquet                   119.5 MB
  03TIC_1145_PV

## 2. Alarm Tags & Data File Mapping

Define the 25 `(alarm_tag, condition)` pairs under analysis and map each to its combined-events parquet file.

In [20]:
alarm_tags = [
    ['03PIC_1620', 'PVHI'],
    ['03FIC_1668', 'PVHI'],
    ['03PI_1655', 'PVHI'],
    ['03TIC_1635', 'PVHI'],
    ['03TIC_1009', 'PVHI'],
    ['03TIC_1745A', 'PVHI'],
    ['03LIC_1608', 'PVHI'],
    ['03PIC_1013', 'PVLO'],
    ['03LIC_1608', 'PVLO'],
    ['03PIC_1023', 'PVLO'],
    ['03TIC_1145', 'PVLO'],
    ['03LIC_1071', 'PVHI'],
    ['03LIC_1016', 'PVHI'],
    ['03PIC_1104', 'PVHI'],
    ['03TIC_1009', 'PVLO'],
    ['03LIC_1619', 'PVLO'],
    ['03TIC_1635', 'PVLO'],
    ['03PIC_1023', 'PVHI'],
    ['03LIC_1619', 'PVHI'],
    ['03PI_1814', 'PVHI'],
    ['03TIC_1023', 'PVLO'],
    ['03TIC_1023', 'PVHI'],
    ['03LIC_1016', 'PVLO'],
    ['03LIC_1071', 'PVLO'],
    ['03TI_1081', 'PVHI'],
]


In [21]:
# ── Build (alarm_tag, condition) → combined_events_file mapping ────────────────
# Tag names in filenames may omit underscores (e.g. 03PIC1023 vs 03PIC_1023),
# so we normalise by stripping underscores before matching.

def normalize_tag(tag):
    return tag.replace('_', '').upper()

# Parse every combined_events filename into {normalized_tag: {alarm_type: Path}}
_file_index = {}
for fp in available_files:
    stem = fp.stem.replace('_combined_events', '')
    for alarm_type in ('PVLO_PVHI', 'PVLO', 'PVHI', 'AOA'):
        if stem.endswith('_' + alarm_type):
            tag_part = stem[: -len('_' + alarm_type)]
            _file_index.setdefault(normalize_tag(tag_part), {})[alarm_type] = fp
            break

tag_combined_file_map = {}   # (alarm_tag, condition) → Path
unmapped_tags = []

for tag, condition in alarm_tags:
    norm = normalize_tag(tag)
    if norm in _file_index:
        fm = _file_index[norm]
        if condition in fm:
            tag_combined_file_map[(tag, condition)] = fm[condition]
        elif 'PVLO_PVHI' in fm:                  # PVLO_PVHI file covers both
            tag_combined_file_map[(tag, condition)] = fm['PVLO_PVHI']
        else:
            unmapped_tags.append((tag, condition))
    else:
        unmapped_tags.append((tag, condition))

print(f"Mapped   : {len(tag_combined_file_map)} / {len(alarm_tags)}")
print(f"Unmapped : {len(unmapped_tags)}\n")

print(f"{'Tag':<16} | {'Cond':<4} | File")
print('-' * 85)
for (tag, cond), fp in tag_combined_file_map.items():
    print(f"  {tag:<16} | {cond:<4} | {fp.name}")

if unmapped_tags:
    print('\nUNMAPPED (no combined_events file available):')
    for tag, cond in unmapped_tags:
        print(f'  {tag}  {cond}')


Mapped   : 22 / 25
Unmapped : 3

Tag              | Cond | File
-------------------------------------------------------------------------------------
  03FIC_1668       | PVHI | 03FIC_1668_PVHI_combined_events.parquet
  03PI_1655        | PVHI | 03PI_1655_PVHI_combined_events.parquet
  03TIC_1635       | PVHI | 03TIC_1635_PVHI_combined_events.parquet
  03TIC_1009       | PVHI | 03TIC_1009_PVHI_combined_events.parquet
  03TIC_1745A      | PVHI | 03TIC_1745A_PVHI_combined_events.parquet
  03PIC_1013       | PVLO | 03PIC_1013_PVLO_combined_events.parquet
  03LIC_1608       | PVLO | 03LIC1608_PVLO_combined_events.parquet
  03PIC_1023       | PVLO | 03PIC1023_PVLO_PVHI_combined_events.parquet
  03TIC_1145       | PVLO | 03TIC_1145_PVLO_combined_events.parquet
  03LIC_1071       | PVHI | 03LIC_1071_PVLO_PVHI_combined_events.parquet
  03LIC_1016       | PVHI | 03LIC_1016_PVLO_PVHI_combined_events.parquet
  03PIC_1104       | PVHI | 03PIC_1104_PVHI_combined_events.parquet
  03TIC_1009       | 

## 3. Loading & Validating Combined Events Data

Cache each combined-events parquet (slim 8-column read), inspect schema, and verify alarm event counts per `(tag, condition)` pair.

In [22]:

# ── Only these 8 columns are ever used downstream ─────────────────────────────
NEEDED_COLS = ['Source', 'ConditionName', 'Category', 'Action',
               'VT_Start', 'Description', 'Value', 'PrevValue']

# ── Load and cache combined_events files (slim columns only) ──────────────────
# Loading all 48 columns expands each file ~50x in RAM (e.g. 147 MB → 7.4 GB).
# Reading only the 8 needed columns keeps each file under ~200 MB.
_combined_events_cache = {}   # str(Path) → DataFrame

def get_combined_events(file_path):
    """Load (and cache) a combined_events parquet file — slim columns only."""
    key = str(file_path)
    if key not in _combined_events_cache:
        # Read only the columns we actually need
        import pyarrow.parquet as pq
        available = pq.read_schema(file_path).names
        cols = [c for c in NEEDED_COLS if c in available]
        df = pd.read_parquet(file_path, columns=cols)
        df['VT_Start'] = pd.to_datetime(df['VT_Start'])
        _combined_events_cache[key] = df
    return _combined_events_cache[key]

# ── Verify alarm event counts per mapped (tag, condition) ─────────────────────
print(f"{'Tag':<16} | {'Cond':<4} | {'Alarm Events':>12} | File")
print('-' * 90)

for (tag, cond), fp in tag_combined_file_map.items():
    df = get_combined_events(fp)
    mask = (df['Source'] == tag) & (df['ConditionName'] == cond)
    if 'Category' in df.columns:
        mask = mask & (df['Category'] == 1)
    count = mask.sum()
    print(f"  {tag:<16} | {cond:<4} | {count:>12,} | {fp.name}")


Tag              | Cond | Alarm Events | File
------------------------------------------------------------------------------------------
  03FIC_1668       | PVHI |           50 | 03FIC_1668_PVHI_combined_events.parquet
  03PI_1655        | PVHI |           20 | 03PI_1655_PVHI_combined_events.parquet
  03TIC_1635       | PVHI |           20 | 03TIC_1635_PVHI_combined_events.parquet
  03TIC_1009       | PVHI |           12 | 03TIC_1009_PVHI_combined_events.parquet
  03TIC_1745A      | PVHI |           18 | 03TIC_1745A_PVHI_combined_events.parquet
  03PIC_1013       | PVLO |          120 | 03PIC_1013_PVLO_combined_events.parquet
  03LIC_1608       | PVLO |           52 | 03LIC1608_PVLO_combined_events.parquet
  03PIC_1023       | PVLO |          194 | 03PIC1023_PVLO_PVHI_combined_events.parquet
  03TIC_1145       | PVLO |          184 | 03TIC_1145_PVLO_combined_events.parquet
  03LIC_1071       | PVHI |          650 | 03LIC_1071_PVLO_PVHI_combined_events.parquet
  03LIC_1016       | PVHI

In [23]:
# Sample combined_events file — show schema
_sample_fp = list(tag_combined_file_map.values())[0]
_df_sample = get_combined_events(_sample_fp)
print(f"File   : {_sample_fp.name}")
print(f"Shape  : {_df_sample.shape}")
print(f"Columns: {_df_sample.columns.tolist()}")
if 'source_table' in _df_sample.columns:
    print(f"\nsource_table breakdown:")
    print(_df_sample['source_table'].value_counts().to_string())
_df_sample.head(3)


File   : 03FIC_1668_PVHI_combined_events.parquet
Shape  : (4005637, 8)
Columns: ['Source', 'ConditionName', 'Category', 'Action', 'VT_Start', 'Description', 'Value', 'PrevValue']


,Source,ConditionName,Category,Action,VT_Start,Description,Value,PrevValue
0,SI_2K101_BN_LCHK,OPERATOR MESSAGE,4.0,None,2021-02-01 15:49:41.804300,2K101_BL: LINK B TO 2K101_BN OK,None,None
1,SI_2K101_BN_LCHK,OPERATOR MESSAGE,4.0,None,2021-02-01 15:49:41.804300,2K101_BL: LINK B TO 2K101_BN OK,None,None
2,None,None,NaN,None,2021-02-04 15:27:51.852700,3C153C PRES,- 0.057,None


## 4. PV/OP Data — Missing Value Analysis

For each parquet file in `DATA/PV-OP_data/`, report row count, date range, and per-column missing value counts/percentages.

In [24]:
# ── PV/OP Data: Missing Value Analysis ────────────────────────────────────────
PV_OP_DATA_DIR = DATA_DIR / 'PV-OP_data'

pv_op_files = sorted(PV_OP_DATA_DIR.glob('*.parquet'))
print(f"Found {len(pv_op_files)} PV/OP parquet file(s) in {PV_OP_DATA_DIR.name}/\n")

for fp in pv_op_files:
    df = pd.read_parquet(fp)
    total = len(df)

    # Date range — handle TimeStamp as column or as index
    if 'TimeStamp' in df.columns:
        ts_min, ts_max = df['TimeStamp'].min(), df['TimeStamp'].max()
    elif df.index.name == 'TimeStamp':
        ts_min, ts_max = df.index.min(), df.index.max()
    else:
        ts_min = ts_max = 'N/A'

    missing_counts = df.isnull().sum()
    missing_pcts   = (missing_counts / total * 100).round(2)

    summary = pd.DataFrame({
        'column':        missing_counts.index,
        'missing_count': missing_counts.values,
        'missing_pct_%': missing_pcts.values,
    })
    cols_with_missing = summary[summary['missing_count'] > 0]

    print(f"{'─' * 65}")
    print(f"  File       : {fp.name}")
    print(f"  Rows       : {total:,}   |   Columns: {len(df.columns)}")
    print(f"  Date range : {ts_min}  →  {ts_max}")

    if cols_with_missing.empty:
        print("  Missing    : none ✓")
    else:
        print(f"  Missing    : {len(cols_with_missing)} column(s) with gaps\n")
        print(cols_with_missing.to_string(index=False))
    print()


Found 9 PV/OP parquet file(s) in PV-OP_data/

─────────────────────────────────────────────────────────────────
  File       : 03FIC_1668_JAN_2026.parquet
  Rows       : 1,737,590   |   Columns: 29
  Date range : 2022-01-03 22:45:00  →  2025-06-23 20:44:00
  Missing    : 26 column(s) with gaps

       column  missing_count  missing_pct_%
03FIC_1668.OP            133           0.01
03FIC_1668.PV             32           0.00
 03FI_1793.PV            254           0.01
03LIC_1603.OP            133           0.01
03LIC_1603.PV             16           0.00
03LIC_1608.OP             13           0.00
03LIC_1608.PV             13           0.00
03LIC_1618.OP            133           0.01
03LIC_1618.PV             13           0.00
03LIC_1619.OP             13           0.00
03LIC_1619.PV             13           0.00
 03LI_1654.PV            290           0.02
03PDI_1611.PV         527927          30.38
03PIC_1068.OP            136           0.01
03PIC_1068.PV             12           0.00


## 5. Alarm Episode Extraction

Helper functions for loading events and pairing Start → OK events into episodes. Iterates all 25 `(tag, condition)` pairs, prints episode counts and orphan stats, and filters to pairs with > 15 alarms.

In [25]:
def load_tag_events(tag, condition):
    """Load alarm events for (tag, condition) from the mapped combined_events file."""
    if (tag, condition) not in tag_combined_file_map:
        return pd.DataFrame()
    df = get_combined_events(tag_combined_file_map[(tag, condition)])
    mask = (df['Source'] == tag) & (df['ConditionName'] == condition)
    if 'Category' in df.columns:
        mask = mask & (df['Category'] == 1)
    return df[mask].copy().sort_values('VT_Start').reset_index(drop=True)


def extract_alarms(tag, condition):
    """Extract alarm episodes (start→end pairs) and log orphans."""
    df = load_tag_events(tag, condition)
    if df.empty:
        return [], [], []

    is_start = df['Action'].isna() | (df['Action'] == '')
    is_end   = df['Action'] == 'OK'

    events_ordered = df[is_start | is_end].copy()
    events_ordered['is_start'] = (
        events_ordered['Action'].isna() | (events_ordered['Action'] == '')
    )
    events_ordered = events_ordered.sort_values('VT_Start').reset_index(drop=True)

    alarms, orphan_starts, orphan_ends = [], [], []
    pending_start = None

    for _, row in events_ordered.iterrows():
        if row['is_start']:
            if pending_start is not None:
                orphan_starts.append(pending_start)
            pending_start = row
        else:
            if pending_start is not None:
                alarms.append({
                    'tag': tag,
                    'condition': condition,
                    'alarm_start': pending_start['VT_Start'],
                    'alarm_end': row['VT_Start'],
                    'duration_minutes': (
                        row['VT_Start'] - pending_start['VT_Start']
                    ).total_seconds() / 60,
                })
                pending_start = None
            else:
                orphan_ends.append(row)

    if pending_start is not None:
        orphan_starts.append(pending_start)

    return alarms, orphan_starts, orphan_ends


# ── Extract alarms for all alarm_tags entries ─────────────────────────────────
all_alarms, all_orphan_starts, all_orphan_ends = [], [], []

print(f"{'Tag':<16} | {'Cond':<4} | {'Alarms':>6} | {'Orphan↑':>8} | {'Orphan↓':>8}")
print('-' * 60)

for tag, condition in alarm_tags:
    if (tag, condition) not in tag_combined_file_map:
        print(f"  {tag:<16} | {condition:<4} |   [NO FILE]")
        continue
    alarms, orphan_starts, orphan_ends = extract_alarms(tag, condition)
    all_alarms.extend(alarms)
    all_orphan_starts.extend(
        {'tag': tag, 'condition': condition, 'timestamp': r['VT_Start']}
        for r in orphan_starts
    )
    all_orphan_ends.extend(
        {'tag': tag, 'condition': condition, 'timestamp': r['VT_Start']}
        for r in orphan_ends
    )
    print(f"  {tag:<16} | {condition:<4} | {len(alarms):>6} | {len(orphan_starts):>8} | {len(orphan_ends):>8}")

print('-' * 60)
print(f"  {'TOTAL':<16} | {'':4} | {len(all_alarms):>6} | {len(all_orphan_starts):>8} | {len(all_orphan_ends):>8}")

alarms_df = pd.DataFrame(all_alarms)
orphan_starts_df = (pd.DataFrame(all_orphan_starts) if all_orphan_starts
                    else pd.DataFrame(columns=['tag', 'condition', 'timestamp']))
orphan_ends_df   = (pd.DataFrame(all_orphan_ends) if all_orphan_ends
                    else pd.DataFrame(columns=['tag', 'condition', 'timestamp']))

print(f"\nAlarms DataFrame shape: {alarms_df.shape}")
print(f"Orphan starts: {len(orphan_starts_df)}   Orphan ends: {len(orphan_ends_df)}")
alarms_df.head(10)


Tag              | Cond | Alarms |  Orphan↑ |  Orphan↓
------------------------------------------------------------
  03PIC_1620       | PVHI |   [NO FILE]
  03FIC_1668       | PVHI |     25 |        0 |        0
  03PI_1655        | PVHI |     10 |        0 |        0
  03TIC_1635       | PVHI |     10 |        0 |        0
  03TIC_1009       | PVHI |      6 |        0 |        0
  03TIC_1745A      | PVHI |      9 |        0 |        0
  03LIC_1608       | PVHI |   [NO FILE]
  03PIC_1013       | PVLO |     55 |       10 |        0
  03LIC_1608       | PVLO |     20 |       12 |        0
  03PIC_1023       | PVLO |     89 |       16 |        0
  03TIC_1145       | PVLO |     89 |        6 |        0
  03LIC_1071       | PVHI |    324 |        2 |        0
  03LIC_1016       | PVHI |    366 |        0 |        0
  03PIC_1104       | PVHI |    302 |        0 |        0
  03TIC_1009       | PVLO |    240 |       12 |        0
  03LIC_1619       | PVLO |    702 |       12 |        0
  03TI

,tag,condition,alarm_start,alarm_end,duration_minutes
0,03FIC_1668,PVHI,2021-12-07 02:28:58.902700,2021-12-07 02:29:05.903600,0.116682
1,03FIC_1668,PVHI,2022-03-03 01:22:59.353500,2022-03-03 01:23:01.353500,0.033333
2,03FIC_1668,PVHI,2022-03-03 08:04:09.852400,2022-03-03 08:04:11.854300,0.033365
3,03FIC_1668,PVHI,2022-03-24 22:05:49.303600,2022-03-24 22:09:31.303700,3.700002
4,03FIC_1668,PVHI,2022-03-24 23:59:58.405300,2022-03-25 00:00:02.405300,0.066667
5,03FIC_1668,PVHI,2022-04-18 17:46:30.601800,2022-04-18 17:46:33.606400,0.050077
6,03FIC_1668,PVHI,2022-08-06 09:04:42.702500,2022-08-06 09:04:43.702500,0.016667
7,03FIC_1668,PVHI,2023-02-03 00:08:07.154400,2023-02-03 00:08:08.154400,0.016667
8,03FIC_1668,PVHI,2023-03-09 10:44:07.152100,2023-03-09 10:44:08.152100,0.016667
9,03FIC_1668,PVHI,2023-06-26 19:02:38.501600,2023-06-26 19:02:41.501600,0.050000


In [26]:
# ── Alarm duration summary per (tag, condition) ────────────────────────────────
print(f"{'Tag':<16} | {'Cond':<4} | {'Count':>6} | {'Mean (min)':>10} | {'Median':>8} | {'Min':>8} | {'Max':>10}")
print('-' * 80)

for tag, condition in alarm_tags:
    eps = alarms_df[(alarms_df['tag'] == tag) & (alarms_df['condition'] == condition)]
    if len(eps) == 0:
        continue
    d = eps['duration_minutes']
    print(f"  {tag:<16} | {condition:<4} | {len(eps):>6} | "
          f"{d.mean():>10.1f} | {d.median():>8.1f} | {d.min():>8.1f} | {d.max():>10.1f}")


Tag              | Cond |  Count | Mean (min) |   Median |      Min |        Max
--------------------------------------------------------------------------------
  03FIC_1668       | PVHI |     25 |        0.3 |      0.1 |      0.0 |        3.7
  03PI_1655        | PVHI |     10 |      195.4 |     12.9 |      2.8 |     1094.4
  03TIC_1635       | PVHI |     10 |        9.9 |      9.7 |      1.0 |       16.8
  03TIC_1009       | PVHI |      6 |       13.0 |      2.4 |      0.6 |       39.8
  03TIC_1745A      | PVHI |      9 |        2.4 |      2.2 |      0.3 |        6.3
  03PIC_1013       | PVLO |     55 |      150.6 |      1.4 |      0.3 |     2515.6
  03LIC_1608       | PVLO |     20 |      332.0 |      1.8 |      0.9 |     6601.2
  03PIC_1023       | PVLO |     89 |      184.8 |     18.0 |      0.1 |     5583.5
  03TIC_1145       | PVLO |     89 |      121.9 |      4.6 |      0.0 |     4636.1
  03LIC_1071       | PVHI |    324 |       33.5 |      0.6 |      0.0 |     1600.5
  03LIC_

In [27]:
# ── Filter alarm tags with more than 15 alarms ──────────────────────
alarm_counts = alarms_df.groupby(['tag', 'condition']).size().reset_index(name='alarm_count')
significant_alarms = alarm_counts[alarm_counts['alarm_count'] > 15]
print(f"Tags with >15 alarms: {len(significant_alarms)} / {len(alarm_counts)}")
print(significant_alarms.to_string(index=False))

# Vectorized filter — avoids slow row-by-row apply lambda
alarms_significant = alarms_df.merge(
    significant_alarms[['tag', 'condition']],
    on=['tag', 'condition'],
    how='inner'
).reset_index(drop=True)
print(f"\nTotal alarm episodes to analyze: {len(alarms_significant)}")


Tags with >15 alarms: 17 / 21
       tag condition  alarm_count
03FIC_1668      PVHI           25
03LIC_1016      PVHI          366
03LIC_1016      PVLO         3758
03LIC_1071      PVHI          324
03LIC_1071      PVLO         1846
03LIC_1608      PVLO           20
03LIC_1619      PVHI          829
03LIC_1619      PVLO          702
03PIC_1013      PVLO           55
03PIC_1023      PVHI          363
03PIC_1023      PVLO           89
03PIC_1104      PVHI          302
03TIC_1009      PVLO          240
03TIC_1023      PVHI          566
03TIC_1023      PVLO          981
03TIC_1145      PVLO           89
03TIC_1635      PVLO          540

Total alarm episodes to analyze: 11095


## 6. Control Actions Extraction

Pull all operator CHANGE events (Category = 7) within a **240-minute pre-alarm** and **60-minute post-alarm** window around each significant alarm episode. Group by source file to avoid redundant parquet loads.

In [28]:
# ── Extract control actions (CHANGE events) around each alarm episode ──────────
# Previously this pre-loaded ALL 17 files into memory simultaneously (~75 GB RAM).
# Now we process one file at a time and release it before loading the next.
#
# NOTE: In these combined_events parquet files:
#   - Alarm events (PVLO/PVHI)  → Category == 1
#   - Operator CHANGE events     → Category == 7  (NOT 1 — was a silent bug)
WINDOW_BEFORE = pd.Timedelta(minutes=240)
WINDOW_AFTER  = pd.Timedelta(minutes=60)

control_actions_list = []

# Group significant alarms by their source file so we only load each file once
alarms_by_file = {}
for idx, alarm in alarms_significant.iterrows():
    key = (alarm['tag'], alarm['condition'])
    if key not in tag_combined_file_map:
        continue
    fp_key = str(tag_combined_file_map[key])
    alarms_by_file.setdefault(fp_key, []).append((idx, alarm))

print(f"Processing {len(alarms_by_file)} unique file(s) for {len(alarms_significant)} alarm episodes...\n")

for fp_key, alarm_list in alarms_by_file.items():
    fp = Path(fp_key)
    df = get_combined_events(fp)

    # CHANGE events live in Category=7 in these combined_events files
    changes = df[(df['ConditionName'] == 'CHANGE') & (df['Category'] == 7)].copy()
    changes = changes.sort_values('VT_Start').reset_index(drop=True)

    print(f"  {fp.name}: {len(changes):,} CHANGE events → processing {len(alarm_list)} alarm(s)")

    for idx, alarm in alarm_list:
        window_start = alarm['alarm_start'] - WINDOW_BEFORE
        window_end   = alarm['alarm_end']   + WINDOW_AFTER

        mask = (changes['VT_Start'] >= window_start) & (changes['VT_Start'] <= window_end)
        actions_in_window = changes[mask].copy()

        if len(actions_in_window) > 0:
            actions_in_window['alarm_tag']       = alarm['tag']
            actions_in_window['alarm_condition'] = alarm['condition']
            actions_in_window['alarm_start']     = alarm['alarm_start']
            actions_in_window['alarm_end']       = alarm['alarm_end']
            actions_in_window['alarm_idx']       = idx
            actions_in_window['source_file']     = fp.name

            actions_in_window['timing'] = 'during'
            actions_in_window.loc[actions_in_window['VT_Start'] < alarm['alarm_start'], 'timing'] = 'before'
            actions_in_window.loc[actions_in_window['VT_Start'] > alarm['alarm_end'],   'timing'] = 'after'

            control_actions_list.append(actions_in_window)

    del changes   # release memory before loading next file

control_actions_df = (pd.concat(control_actions_list, ignore_index=True)
                      if control_actions_list else pd.DataFrame())

print(f"\nTotal control actions extracted : {len(control_actions_df):,}")
if len(control_actions_df) > 0:
    print(f"Unique operated tags (Source)   : {control_actions_df['Source'].nunique()}")
    print(f"\nTiming breakdown:")
    print(control_actions_df['timing'].value_counts())
    print(f"\nSource file breakdown:")
    print(control_actions_df['source_file'].value_counts())


Processing 12 unique file(s) for 11095 alarm episodes...

  03FIC_1668_PVHI_combined_events.parquet: 290,298 CHANGE events → processing 25 alarm(s)
  03PIC_1013_PVLO_combined_events.parquet: 163,254 CHANGE events → processing 55 alarm(s)
  03LIC1608_PVLO_combined_events.parquet: 140,394 CHANGE events → processing 20 alarm(s)
  03PIC1023_PVLO_PVHI_combined_events.parquet: 257,414 CHANGE events → processing 452 alarm(s)
  03TIC_1145_PVLO_combined_events.parquet: 201,692 CHANGE events → processing 89 alarm(s)
  03LIC_1071_PVLO_PVHI_combined_events.parquet: 389,646 CHANGE events → processing 2170 alarm(s)
  03LIC_1016_PVLO_PVHI_combined_events.parquet: 201,692 CHANGE events → processing 4124 alarm(s)
  03PIC_1104_PVHI_combined_events.parquet: 412,002 CHANGE events → processing 302 alarm(s)
  03TIC_1009_PVLO_combined_events.parquet: 215,646 CHANGE events → processing 240 alarm(s)
  03LIC_1619_PVLO_PVHI_combined_events.parquet: 80,994 CHANGE events → processing 1531 alarm(s)
  03TIC_1635_PVL

In [29]:
control_actions_df.columns

Index(['Source', 'ConditionName', 'Category', 'Action', 'VT_Start',
       'Description', 'Value', 'PrevValue', 'alarm_tag', 'alarm_condition',
       'alarm_start', 'alarm_end', 'alarm_idx', 'source_file', 'timing'],
      dtype='object')

## 7. Control Actions Analysis

Classify extracted CHANGE events by action type (OP / SP / MODE), identify the most frequently operated tags per alarm, and summarise overall timing patterns.

In [30]:
# ── Classify control action types from Description column ──────────
# Description contains values like 'OP', 'SP', 'MODE', 'SO', 'PVFL', etc.
def classify_action_type(desc):
    """Classify control action type from Description field."""
    if pd.isna(desc):
        return 'OTHER'
    desc_str = str(desc).strip().upper()
    if desc_str == 'OP':
        return 'OP'
    elif desc_str == 'SP':
        return 'SP'
    elif desc_str == 'MODE':
        return 'MODE'
    else:
        return 'OTHER'

control_actions_df['action_type'] = control_actions_df['Description'].apply(classify_action_type)

print("Control action type breakdown:")
print(control_actions_df['action_type'].value_counts())
print(f"\n{'='*70}")

# Filter to only OP, SP, MODE actions
control_actions_relevant = control_actions_df[
    control_actions_df['action_type'].isin(['OP', 'SP', 'MODE'])
].copy()
print(f"\nRelevant control actions (OP/SP/MODE): {len(control_actions_relevant):,} / {len(control_actions_df):,}")

# ── Which tags were operated most for each alarm tag? ──────────────
print("\n\nTop operated tags (Source) per alarm tag/condition:")
print("=" * 70)

for (alarm_tag, alarm_cond), grp in control_actions_relevant.groupby(['alarm_tag', 'alarm_condition']):
    n_alarms = len(alarms_significant[
        (alarms_significant['tag'] == alarm_tag) & (alarms_significant['condition'] == alarm_cond)
    ])
    print(f"\n{alarm_tag} | {alarm_cond} ({n_alarms} alarms, {len(grp)} OP/SP/MODE actions)")
    print("-" * 50)
    
    # Top operated tags with action type breakdown
    top_tags = grp.groupby(['Source', 'action_type']).size().unstack(fill_value=0)
    top_tags['total'] = top_tags.sum(axis=1)
    top_tags = top_tags.sort_values('total', ascending=False).head(10)
    print(top_tags.to_string())

Control action type breakdown:
action_type
OP       2237566
SP        322052
MODE      118498
OTHER      83812
Name: count, dtype: int64


Relevant control actions (OP/SP/MODE): 2,678,116 / 2,761,928


Top operated tags (Source) per alarm tag/condition:

03FIC_1668 | PVHI (25 alarms, 13990 OP/SP/MODE actions)
--------------------------------------------------
action_type  MODE    OP   SP  total
Source                             
03FIC_1668      0  1618    0   1618
03TIC_1635     66  1382  118   1566
02HIC_1087      0  1326    0   1326
02HIC_1050      0  1050    0   1050
03LIC_1016     24   926   20    970
04RES_3F102     0   942    0    942
03LIC_1619     30   860   52    942
02FIC_1247      0   898    0    898
03LIC_1608     38   684  128    850
03ESDV_1669     0   518    0    518

03LIC_1016 | PVHI (366 alarms, 221238 OP/SP/MODE actions)
--------------------------------------------------
action_type  MODE     OP    SP  total
Source                               
03HIC_1141    376  3

In [31]:
# ── Summary: action types and top operated tags across all alarm tags ───
print("Overall action type breakdown:")
print(control_actions_df['action_type'].value_counts())
print(f"\nRelevant (OP/SP/MODE): {len(control_actions_relevant):,}")
print(f"\n{'='*70}")
print("\nTop 15 most operated tags across ALL alarm episodes (OP/SP/MODE only):")
overall_top = control_actions_relevant.groupby(['Source', 'action_type']).size().unstack(fill_value=0)
overall_top['total'] = overall_top.sum(axis=1)
overall_top = overall_top.sort_values('total', ascending=False).head(15)
print(overall_top.to_string())

print(f"\n{'='*70}")
print("\nTiming breakdown for OP/SP/MODE actions:")
print(control_actions_relevant.groupby(['timing', 'action_type']).size().unstack(fill_value=0))

Overall action type breakdown:
action_type
OP       2237566
SP        322052
MODE      118498
OTHER      83812
Name: count, dtype: int64

Relevant (OP/SP/MODE): 2,678,116


Top 15 most operated tags across ALL alarm episodes (OP/SP/MODE only):
action_type   MODE      OP     SP   total
Source                                   
03HIC_1141    3926  290348      0  294274
03HIC_1151    5724  252734      0  258458
02HIC_1087    4044  214074      0  218118
03PIC_1013    4824  208950      0  213774
02HIC_1050    4194  175716      0  179910
03LIC_1071    9092  118678  18450  146220
03LIC_1016   10288   94400  25974  130662
03PIC_1068    1860    1200  98112  101172
02FIC_1247    2404   87916      0   90320
03LIC_1619    4968   65616   9638   80222
03LIC_1034    7068   12312  59906   79286
03FIC_1085   15886   47460    392   63738
03FIC_3435    1352   48944     10   50306
04RES_2K101      0   44074      0   44074
03PIC_3131    1338   36292   2580   40210


Timing breakdown for OP/SP/MODE actions:

## 8. Data Coverage — Timestamp Ranges

Report the temporal span (min/max `VT_Start`) and row count of each unique combined-events file in use.

In [32]:
# ── Timestamp ranges for each combined_events file in use ─────────────────────
print(f"{'File':<60} | {'Min VT_Start':<26} | {'Max VT_Start':<26} | {'Rows':>8}")
print('-' * 130)

seen = set()
for (tag, cond), fp in sorted(tag_combined_file_map.items()):
    if str(fp) in seen:
        continue
    seen.add(str(fp))
    df_tmp = get_combined_events(fp)
    print(f"{fp.name:<60} | {str(df_tmp['VT_Start'].min()):<26} | "
          f"{str(df_tmp['VT_Start'].max()):<26} | {len(df_tmp):>8,}")


File                                                         | Min VT_Start               | Max VT_Start               |     Rows
----------------------------------------------------------------------------------------------------------------------------------
03FIC_1668_PVHI_combined_events.parquet                      | 2021-02-01 15:49:41.804300 | 2025-06-28 03:57:42.753600 | 4,005,637
03LIC_1016_PVLO_PVHI_combined_events.parquet                 | 2021-02-01 15:49:41.804300 | 2025-06-27 22:52:59.137300 | 4,401,453
03LIC_1071_PVLO_PVHI_combined_events.parquet                 | 2021-02-01 15:49:41.804300 | 2025-06-28 01:56:39.783700 | 5,573,157
03LIC1608_PVLO_combined_events.parquet                       | 2021-02-01 15:49:41.804300 | 2025-06-28 03:57:42.753600 | 3,270,820
03LIC_1619_PVLO_PVHI_combined_events.parquet                 | 2021-02-01 15:49:41.804300 | 2025-06-28 03:57:42.753600 | 3,021,947
03PIC_1013_PVLO_combined_events.parquet                      | 2021-09-06 12:33:15.3

## 9. Export to Excel

Saves two Excel workbooks to `RESULTS/all_alarm_tags_analysis/`:

| File | Sheets | Content |
|------|--------|---------|
| `pvop_missing_values.xlsx` | One sheet per PV/OP parquet file | `column`, `missing_count`, `missing_pct_%` — only columns with gaps |
| `control_actions_analysis.xlsx` | One sheet per `(alarm_tag, condition)` pair | `Source`, `MODE`, `OP`, `SP`, `total` — top operated tags sorted by total |

In [33]:
EXPORT_DIR = PROJECT_ROOT / 'RESULTS' / 'all_alarm_tags_analysis'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
# 1. pvop_missing_values.xlsx  — one sheet per PV/OP parquet file
#    Columns: column | missing_count | missing_pct_%
#    Only rows where missing_count > 0 are included.
# ══════════════════════════════════════════════════════════════════════════════
out1 = EXPORT_DIR / 'pvop_missing_values.xlsx'
with pd.ExcelWriter(out1, engine='openpyxl') as writer:
    for fp in pv_op_files:
        df = pd.read_parquet(fp)
        total = len(df)

        missing_counts = df.isnull().sum()
        missing_pcts   = (missing_counts / total * 100).round(2)

        sheet_df = pd.DataFrame({
            'column':        missing_counts.index,
            'missing_count': missing_counts.values,
            'missing_pct_%': missing_pcts.values,
        })
        sheet_df = sheet_df[sheet_df['missing_count'] > 0].reset_index(drop=True)

        # Derive a clean sheet name from the filename (strip path/extension, max 31 chars)
        sheet_name = fp.stem.replace('_JAN_2026', '').replace('_MAY_2026', '')
        sheet_name = sheet_name.replace('_filtered', '')[:31]

        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"  {sheet_name:<30s}  {len(sheet_df)} column(s) with missing data")

print(f"✓  {out1.name}\n")


# ══════════════════════════════════════════════════════════════════════════════
# 2. control_actions_analysis.xlsx  — one sheet per (alarm_tag, condition) pair
#    Columns: Source | MODE | OP | SP | total  (sorted by total desc)
# ══════════════════════════════════════════════════════════════════════════════
out2 = EXPORT_DIR / 'control_actions_analysis.xlsx'
with pd.ExcelWriter(out2, engine='openpyxl') as writer:
    for (alarm_tag, alarm_cond), grp in control_actions_relevant.groupby(
        ['alarm_tag', 'alarm_condition']
    ):
        n_alarms = len(
            alarms_significant[
                (alarms_significant['tag'] == alarm_tag)
                & (alarms_significant['condition'] == alarm_cond)
            ]
        )

        top_tags = (
            grp.groupby(['Source', 'action_type'])
            .size()
            .unstack(fill_value=0)
            .reset_index()
        )
        # Ensure all three action-type columns always exist
        for col in ['MODE', 'OP', 'SP']:
            if col not in top_tags.columns:
                top_tags[col] = 0
        top_tags = top_tags[['Source', 'MODE', 'OP', 'SP']]
        top_tags['total'] = top_tags[['MODE', 'OP', 'SP']].sum(axis=1)
        top_tags = top_tags.sort_values('total', ascending=False).reset_index(drop=True)

        # Sheet name: e.g. "03FIC_1668_PVHI" (max 31 chars)
        sheet_name = f"{alarm_tag}_{alarm_cond}"[:31]

        top_tags.to_excel(writer, sheet_name=sheet_name, index=False)
        print(f"  {sheet_name:<25s}  {n_alarms} alarms  |  {len(top_tags)} operated tags")

print(f"✓  {out2.name}")
print(f"\nAll files saved to: {EXPORT_DIR}")


  03FIC_1668                      26 column(s) with missing data
  03LIC_1016                      33 column(s) with missing data
  03LIC_1071                      43 column(s) with missing data
  03LIC_1071                      43 column(s) with missing data
  03LIC_1619                      34 column(s) with missing data
  03PIC_1104                      37 column(s) with missing data
  03TIC_1009                      23 column(s) with missing data
  03TIC_1023                      34 column(s) with missing data
  03TIC_1635                      27 column(s) with missing data
✓  pvop_missing_values.xlsx

  03FIC_1668_PVHI            25 alarms  |  88 operated tags
  03LIC_1016_PVHI            366 alarms  |  111 operated tags
  03LIC_1016_PVLO            3758 alarms  |  128 operated tags
  03LIC_1071_PVHI            324 alarms  |  180 operated tags
  03LIC_1071_PVLO            1846 alarms  |  224 operated tags
  03LIC_1608_PVLO            20 alarms  |  58 operated tags
  03LIC_1619_PVH

## 10. Feasibility Report — Confluence Tables

Generates three paste-ready tables for the Confluence feasibility page:
1. **Missing data summary** — one row per alarm tag, highest-gap column highlighted
2. **Control actions summary** — one row per `(alarm_tag, condition)`, OP/SP/MODE/Total columns

In [34]:
# ── Table 1: PV/OP Missing Data Summary (one row per unique tag with PV/OP file) ──
print("=" * 72)
print("TABLE 1 — PV/OP Missing Data Summary (paste into Confluence)")
print("=" * 72)
print(f"\n{'Tag':<16} | {'File Coverage':<27} | {'Cols w/ Gaps':>12} | {'Worst Column':<22} | {'Worst Missing %':>15}")
print('-' * 102)

for fp in pv_op_files:
    # Skip filtered variants — the base file already covers the same tag
    if '_filtered' in fp.stem:
        continue

    df_tmp  = pd.read_parquet(fp)
    total   = len(df_tmp)
    mc      = df_tmp.isnull().sum()
    gaps    = mc[mc > 0]
    n_gaps  = len(gaps)
    if n_gaps > 0:
        worst_col = gaps.idxmax()
        worst_pct = round(gaps.max() / total * 100, 2)
    else:
        worst_col = '—'
        worst_pct = 0.0

    # Date range
    if 'TimeStamp' in df_tmp.columns:
        date_range = f"{df_tmp['TimeStamp'].min().date()} → {df_tmp['TimeStamp'].max().date()}"
    elif df_tmp.index.name == 'TimeStamp':
        date_range = f"{df_tmp.index.min().date()} → {df_tmp.index.max().date()}"
    else:
        date_range = 'unknown'

    tag_name = fp.stem.replace('_JAN_2026', '').replace('_MAY_2026', '')
    print(f"  {tag_name:<14} | {date_range:<27} | {n_gaps:>12} | {worst_col:<22} | {worst_pct:>14.2f}%")

print()
print("NOTE: Tags NOT in this list have no PV/OP data file in DATA/PV-OP_data/.")
print("      Full per-column detail is in pvop_missing_values.xlsx")


TABLE 1 — PV/OP Missing Data Summary (paste into Confluence)

Tag              | File Coverage               | Cols w/ Gaps | Worst Column           | Worst Missing %
------------------------------------------------------------------------------------------------------
  03FIC_1668     | 2022-01-03 → 2025-06-23     |           26 | 03PI_1655.PV           |          54.01%
  03LIC_1016     | 2022-01-03 → 2025-06-23     |           33 | 03LIC_1183.PV          |          26.88%
  03LIC_1071     | 2022-01-03 → 2025-06-23     |           43 | 03TIC_1142.PV          |           7.00%
  03LIC_1619     | 2022-01-03 → 2025-06-23     |           34 | 03PI_1655.PV           |          54.01%
  03PIC_1104     | 2022-01-03 → 2025-06-23     |           37 | 03LIC_3178.PV          |           0.64%
  03TIC_1009     | 2022-01-03 → 2025-06-23     |           23 | 03TI_1002.PV           |           0.18%
  03TIC_1023     | 2022-01-03 → 2025-06-23     |           34 | 03TI_3121.PV           |           0

In [17]:
# ── Table 2: Control Actions Summary (alarm-tag level) ────────────────────────
print("=" * 72)
print("TABLE 2 — Control Actions Summary (paste into Confluence)")
print("=" * 72)
print(f"\n{'Tag':<16} | {'Cond':<4} | {'Episodes':>9} | {'OP':>8} | {'SP':>8} | {'MODE':>8} | {'Total':>8}")
print('-' * 75)

ca_summary_rows = []
for tag, condition in alarm_tags:
    n_eps = len(alarms_significant[
        (alarms_significant['tag'] == tag) & (alarms_significant['condition'] == condition)
    ]) if 'alarms_significant' in dir() else 0

    if (tag, condition) not in tag_combined_file_map:
        print(f"  {tag:<16} | {condition:<4} | {'[NO FILE]':>9}")
        ca_summary_rows.append({'tag': tag, 'condition': condition, 'alarm_episodes': None,
                                 'OP': None, 'SP': None, 'MODE': None, 'total_actions': None,
                                 'note': 'no combined_events file'})
        continue

    grp = control_actions_relevant[
        (control_actions_relevant['alarm_tag'] == tag) &
        (control_actions_relevant['alarm_condition'] == condition)
    ]
    op_n   = (grp['action_type'] == 'OP').sum()
    sp_n   = (grp['action_type'] == 'SP').sum()
    mode_n = (grp['action_type'] == 'MODE').sum()
    total  = op_n + sp_n + mode_n

    if n_eps == 0 and total == 0:
        note = 'no alarms / not in significant set'
    else:
        note = ''

    print(f"  {tag:<16} | {condition:<4} | {n_eps:>9,} | {op_n:>8,} | {sp_n:>8,} | {mode_n:>8,} | {total:>8,}")
    ca_summary_rows.append({'tag': tag, 'condition': condition, 'alarm_episodes': n_eps,
                             'OP': op_n, 'SP': sp_n, 'MODE': mode_n, 'total_actions': total, 'note': note})

print('-' * 75)
total_eps = sum(r['alarm_episodes'] or 0 for r in ca_summary_rows)
total_op  = sum(r['OP'] or 0 for r in ca_summary_rows)
total_sp  = sum(r['SP'] or 0 for r in ca_summary_rows)
total_md  = sum(r['MODE'] or 0 for r in ca_summary_rows)
print(f"  {'TOTAL':<16} | {'':4} | {total_eps:>9,} | {total_op:>8,} | {total_sp:>8,} | {total_md:>8,} | {total_op+total_sp+total_md:>8,}")
print()
print("Window: 240 min before alarm start → 60 min after alarm end")
print("Only OP / SP / MODE change events are counted (OTHER types excluded).")
print("Detailed per-operated-tag breakdown: control_actions_analysis.xlsx")


TABLE 2 — Control Actions Summary (paste into Confluence)

Tag              | Cond |  Episodes |       OP |       SP |     MODE |    Total
---------------------------------------------------------------------------
  03PIC_1620       | PVHI | [NO FILE]
  03FIC_1668       | PVHI |        25 |   12,754 |      794 |      442 |   13,990
  03PI_1655        | PVHI |         0 |        0 |        0 |        0 |        0
  03TIC_1635       | PVHI |         0 |        0 |        0 |        0 |        0
  03TIC_1009       | PVHI |         0 |        0 |        0 |        0 |        0
  03TIC_1745A      | PVHI |         0 |        0 |        0 |        0 |        0
  03LIC_1608       | PVHI | [NO FILE]
  03PIC_1013       | PVLO |        55 |   13,348 |    2,724 |      752 |   16,824
  03LIC_1608       | PVLO |        20 |    2,548 |      376 |      222 |    3,146
  03PIC_1023       | PVLO |        89 |   65,194 |    4,318 |    3,030 |   72,542
  03TIC_1145       | PVLO |        89 |   12,876 |   